In [2]:
!pip install tensorrt

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 18.3 MB/s eta 0:00:0000:010:01
  Created wheel for tensorrt: filename=tensorrt-10.16.0.72-py3-none-any.whl size=16566 sha256=9efeb45daa26d3b88a220a0807c5e51870e00e0bd15052650152423e7071e248
  Stored in directory: /root/.cache/pip/wheels/94/5f/04/7f93fac79967299afd72fc4feedfa60dbd27352e553e94582b
  Created wheel for tensorrt_cu13: filename=tensorrt_cu13-10.16.0.72-py3-none-any.whl size=23077 sha256=c55ceb89c890c7d74e57003c453adf13c2b7b73ce623afa7981a7265a1674539
  Stored in directory: /root/.cache/pip/wheels/11/9d/13/e122974a313a392b70

In [3]:
import io
import time
import json
from pathlib import Path

import numpy as np
import librosa
import torch
import torch.nn as nn
import timm
from PIL import Image
from torchvision import transforms

import onnx
import tensorrt as trt
import pycuda.driver as cuda
import pycuda.autoinit  # initializes CUDA context automatically

print(f"PyTorch  : {torch.__version__}")
print(f"TensorRT : {trt.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch  : 2.10.0+cu128
TensorRT : 10.16.0.72
CUDA available: True
GPU: Tesla T4


In [13]:
# --- Paths ---
PTH_PATH    = Path("/kaggle/input/datasets/akhilvaidya/dsfghg/deit_mel_spectrogram.pth")       # your fine-tuned DeiT weights
ONNX_PATH   = Path("/kaggle/working/.onnx")      # intermediate, written by Cell 3
ENGINE_PATH = Path("/kaggle/working/deit_fp16.engine")     # final TRT engine, written by Cell 4
WAV_PATH    = Path("/kaggle/input/datasets/akhilvaidya/dsfghg/1_10101.wav")       # wav file to run inference on
CLASS_NAMES_PATH = Path("class_names.json") # optional - list of class name strings

# --- Model ---
NUM_CLASSES  = 27          # change to match your fine-tuned head
MODEL_ARCH   = "deit_small_patch16_224"

# --- Audio ---
SAMPLE_RATE  = 22050
CLIP_SECONDS = 5
N_MELS       = 128
N_FFT        = 2048
HOP_LENGTH   = 512
IMG_SIZE     = 224

# --- TRT ---
USE_FP16     = True        # recommended on T4
TRT_WORKSPACE_GB = 4

TRT_LOGGER = trt.Logger(trt.Logger.WARNING)

In [22]:
assert PTH_PATH.exists(), f"Model weights not found: {PTH_PATH}"

# Build architecture - replicate exactly what your training code used
model = timm.create_model(MODEL_ARCH, pretrained=False)
model.head = nn.Linear(model.head.in_features, NUM_CLASSES)

state = torch.load(PTH_PATH, map_location="cpu")
model.load_state_dict(state, strict=False)
model.eval().cuda()

dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device="cuda")

torch.onnx.export(
    model,
    dummy,
    str(ONNX_PATH),
    opset_version=17,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes=None,   # fixed shape - better for TRT optimization
)

# Sanity check the exported graph
onnx_model = onnx.load(str(ONNX_PATH))
onnx.checker.check_model(onnx_model)

print(f"ONNX export OK -> {ONNX_PATH}")
print(f"Graph inputs  : {[i.name for i in onnx_model.graph.input]}")
print(f"Graph outputs : {[o.name for o in onnx_model.graph.output]}")

# Free GPU memory - not needed until TRT build
del model, dummy
torch.cuda.empty_cache()

W0404 05:55:06.570000 55 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0404 05:55:07.060000 55 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0404 05:55:07.061000 55 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, ali

[torch.onnx] Obtain model graph for `VisionTransformer([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `VisionTransformer([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
ONNX export OK -> /kaggle/working/.onnx
Graph inputs  : ['input']
Graph outputs : ['output']


In [15]:
assert ONNX_PATH.exists(), f"ONNX file not found - run Cell 3 first: {ONNX_PATH}"

builder = trt.Builder(TRT_LOGGER)
network_flags = 1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH)
network = builder.create_network(network_flags)
parser  = trt.OnnxParser(network, TRT_LOGGER)

with open(ONNX_PATH, "rb") as f:
    raw = f.read()

if not parser.parse(raw):
    for i in range(parser.num_errors):
        print(f"ONNX parse error {i}: {parser.get_error(i)}")
    raise RuntimeError("ONNX parsing failed")

config = builder.create_builder_config()
config.set_memory_pool_limit(
    trt.MemoryPoolType.WORKSPACE,
    TRT_WORKSPACE_GB << 30
)

if USE_FP16 and builder.platform_has_fast_fp16:
    config.set_flag(trt.BuilderFlag.FP16)
    print("FP16 enabled")
else:
    print("FP16 not available on this device - building FP32 engine")

print("Building TRT engine - this will take a few minutes...")
t0 = time.time()

serialized_engine = builder.build_serialized_network(network, config)
if serialized_engine is None:
    raise RuntimeError("TRT engine build failed")

with open(ENGINE_PATH, "wb") as f:
    f.write(serialized_engine)

print(f"Engine saved -> {ENGINE_PATH}  ({time.time()-t0:.1f}s)")
print(f"Engine size  : {ENGINE_PATH.stat().st_size / 1e6:.1f} MB")

del builder, network, parser, config, serialized_engine

FP16 enabled
Building TRT engine - this will take a few minutes...
Engine saved -> /kaggle/working/deit_fp16.engine  (25.6s)
Engine size  : 45.4 MB


In [17]:
assert ENGINE_PATH.exists(), f"Engine file not found - run Cell 4 first: {ENGINE_PATH}"

with open(ENGINE_PATH, "rb") as f, trt.Runtime(TRT_LOGGER) as runtime:
    engine = runtime.deserialize_cuda_engine(f.read())

context = engine.create_execution_context()

# TRT 10.x API - iterate over IO tensors by index
num_io = engine.num_io_tensors
print(f"Engine IO tensors: {num_io}")
for i in range(num_io):
    name  = engine.get_tensor_name(i)
    mode  = engine.get_tensor_mode(name)   # INPUT or OUTPUT
    shape = engine.get_tensor_shape(name)
    dtype = engine.get_tensor_dtype(name)
    print(f"  [{i}] '{name}'  mode={mode}  shape={shape}  dtype={dtype}")

INPUT_NAME  = "input"
OUTPUT_NAME = "output"

input_shape  = (1, 3, IMG_SIZE, IMG_SIZE)
output_shape = (1, NUM_CLASSES)

dtype = np.float16 if USE_FP16 else np.float32

# Pinned host buffers
h_input  = cuda.pagelocked_empty(input_shape,  dtype=dtype)
h_output = cuda.pagelocked_empty(output_shape, dtype=dtype)

# Device buffers - allocated once
d_input  = cuda.mem_alloc(h_input.nbytes)
d_output = cuda.mem_alloc(h_output.nbytes)

stream = cuda.Stream()

# Set tensor addresses on the context once - TRT 10.x replaces bindings list
context.set_tensor_address(INPUT_NAME,  int(d_input))
context.set_tensor_address(OUTPUT_NAME, int(d_output))

print("Buffers allocated, tensor addresses set")

Engine IO tensors: 2
  [0] 'input'  mode=TensorIOMode.INPUT  shape=(1, 3, 224, 224)  dtype=DataType.FLOAT
  [1] 'output'  mode=TensorIOMode.OUTPUT  shape=(1, 27)  dtype=DataType.FLOAT
Buffers allocated, tensor addresses set


In [18]:
NORMALIZE = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


def load_audio(wav_path: Path) -> tuple[np.ndarray, int]:
    y, sr = librosa.load(str(wav_path), sr=SAMPLE_RATE, mono=True)
    if len(y) == 0:
        return np.zeros(SAMPLE_RATE, dtype=np.float32), SAMPLE_RATE
    target_len = SAMPLE_RATE * CLIP_SECONDS
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)), mode="constant")
    else:
        y = y[:target_len]
    return librosa.util.normalize(y), sr


def mel_to_image(y: np.ndarray, sr: int) -> Image.Image:
    """
    Converts raw audio to a mel spectrogram PIL image.
    Uses direct numpy normalization - no matplotlib figure,
    no savefig, no buffer round-trip. ~50-100ms faster per call.
    """
    mel_spec = librosa.feature.melspectrogram(
        y=y, sr=sr, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH
    )
    mel_db = librosa.power_to_db(mel_spec, ref=np.max)  # shape: (N_MELS, T)

    # Normalize to [0, 255] uint8
    mel_min, mel_max = mel_db.min(), mel_db.max()
    mel_norm = ((mel_db - mel_min) / (mel_max - mel_min + 1e-6) * 255).astype(np.uint8)

    # Single channel -> RGB by stacking (matches what the model was trained on)
    mel_rgb = np.stack([mel_norm, mel_norm, mel_norm], axis=-1)  # (N_MELS, T, 3)

    return Image.fromarray(mel_rgb, mode="RGB")


def wav_to_tensor(wav_path: Path) -> torch.Tensor:
    y, sr = load_audio(wav_path)
    image = mel_to_image(y, sr)
    tensor = NORMALIZE(image).unsqueeze(0)  # (1, 3, 224, 224), float32, CPU
    return tensor


def softmax_numpy(logits: np.ndarray) -> np.ndarray:
    e = np.exp(logits - np.max(logits))
    return e / e.sum()


In [23]:
def infer_trt(tensor: torch.Tensor) -> tuple[int, float, np.ndarray]:
    np_input = tensor.numpy().astype(dtype)
    np.copyto(h_input, np_input)

    # Async H2D -> execute -> D2H
    cuda.memcpy_htod_async(d_input, h_input, stream)
    context.execute_async_v3(stream_handle=stream.handle)   # TRT 10.x call
    cuda.memcpy_dtoh_async(h_output, d_output, stream)
    stream.synchronize()

    logits      = h_output[0].astype(np.float32)
    probabilities = softmax_numpy(logits)
    class_index = int(np.argmax(probabilities))
    confidence  = float(probabilities[class_index])

    return class_index, confidence, probabilities

In [20]:
assert WAV_PATH.exists(), f"WAV file not found: {WAV_PATH}"

# Load class names if available, else use generic labels
if CLASS_NAMES_PATH.exists():
    with CLASS_NAMES_PATH.open("r") as f:
        class_names = json.load(f)
else:
    class_names = [f"class_{i}" for i in range(NUM_CLASSES)]

# --- Preprocessing ---
t_pre_start = time.perf_counter()
tensor = wav_to_tensor(WAV_PATH)
t_pre_end = time.perf_counter()
pre_ms = (t_pre_end - t_pre_start) * 1000

# --- TRT Warmup (first call builds CUDA graphs internally) ---
print("Warming up engine...")
for _ in range(3):
    infer_trt(tensor)

# --- Timed Inference ---
N_RUNS = 20
latencies = []
for _ in range(N_RUNS):
    t0 = time.perf_counter()
    class_idx, confidence, probs = infer_trt(tensor)
    latencies.append((time.perf_counter() - t0) * 1000)

infer_ms_mean = np.mean(latencies)
infer_ms_p50  = np.percentile(latencies, 50)
infer_ms_p99  = np.percentile(latencies, 99)

# --- Results ---
print()
print("=" * 45)
print(f"  WAV file       : {WAV_PATH.name}")
print(f"  Predicted class: {class_idx} - {class_names[class_idx]}")
print(f"  Confidence     : {confidence*100:.2f}%")
print("=" * 45)
print(f"  Preprocessing  : {pre_ms:.1f} ms  (librosa + mel + normalize)")
print(f"  TRT inference  : {infer_ms_mean:.2f} ms mean  |  p50={infer_ms_p50:.2f}  p99={infer_ms_p99:.2f}  (n={N_RUNS})")
print(f"  Total latency  : {pre_ms + infer_ms_mean:.1f} ms")
print("=" * 45)
print()
print("Per-class probabilities:")
for i, (name, prob) in enumerate(zip(class_names, probs)):
    bar = "#" * int(prob * 40)
    marker = " <--" if i == class_idx else ""
    print(f"  [{i:2d}] {name:<20s} {prob*100:5.2f}%  {bar}{marker}")

Warming up engine...

  WAV file       : 1_10101.wav
  Predicted class: 22 - class_22
  Confidence     : 100.00%
  Preprocessing  : 13544.4 ms  (librosa + mel + normalize)
  TRT inference  : 4.44 ms mean  |  p50=4.30  p99=6.14  (n=20)
  Total latency  : 13548.9 ms

Per-class probabilities:
  [ 0] class_0               0.00%  
  [ 1] class_1               0.00%  
  [ 2] class_2               0.00%  
  [ 3] class_3               0.00%  
  [ 4] class_4               0.00%  
  [ 5] class_5               0.00%  
  [ 6] class_6               0.00%  
  [ 7] class_7               0.00%  
  [ 8] class_8               0.00%  
  [ 9] class_9               0.00%  
  [10] class_10              0.00%  
  [11] class_11              0.00%  
  [12] class_12              0.00%  
  [13] class_13              0.00%  
  [14] class_14              0.00%  
  [15] class_15              0.00%  
  [16] class_16              0.00%  
  [17] class_17              0.00%  
  [18] class_18              0.00%  
  [19]

/tmp/ipykernel_55/1922951288.py:38: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  return Image.fromarray(mel_rgb, mode="RGB")


In [21]:
assert PTH_PATH.exists(), f"PTH not found: {PTH_PATH}"

pt_model = timm.create_model(MODEL_ARCH, pretrained=False)
pt_model.head = nn.Linear(pt_model.head.in_features, NUM_CLASSES)
state = torch.load(PTH_PATH, map_location="cpu")
pt_model.load_state_dict(state, strict=False)
pt_model.eval().cuda()

cuda_tensor = tensor.cuda()

# Warmup
with torch.no_grad():
    for _ in range(3):
        _ = pt_model(cuda_tensor)
torch.cuda.synchronize()

# Timed runs
pt_latencies = []
with torch.no_grad():
    for _ in range(N_RUNS):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        logits = pt_model(cuda_tensor)
        torch.cuda.synchronize()
        pt_latencies.append((time.perf_counter() - t0) * 1000)

pt_mean = np.mean(pt_latencies)
pt_p50  = np.percentile(pt_latencies, 50)
pt_p99  = np.percentile(pt_latencies, 99)

print()
print("=" * 50)
print("Latency comparison (model inference only, no preprocessing)")
print("-" * 50)
print(f"  PyTorch FP32   : {pt_mean:.2f} ms mean  |  p50={pt_p50:.2f}  p99={pt_p99:.2f}")
print(f"  TRT FP16       : {infer_ms_mean:.2f} ms mean  |  p50={infer_ms_p50:.2f}  p99={infer_ms_p99:.2f}")
print(f"  Speedup        : {pt_mean / infer_ms_mean:.2f}x")
print("=" * 50)

del pt_model, cuda_tensor
torch.cuda.empty_cache()


Latency comparison (model inference only, no preprocessing)
--------------------------------------------------
  PyTorch FP32   : 9.34 ms mean  |  p50=9.33  p99=9.37
  TRT FP16       : 4.44 ms mean  |  p50=4.30  p99=6.14
  Speedup        : 2.11x
